# Dynamic programming: Value aur Policy Iteration

Is section mein, we will apply value aur policy iteration to ek toy environment that consist karta hai ek 3 x 4 grid that's depicted mein following diagram ke saath following features:

- **States**: 11 states represented as two-dimensional coordinates. One field hai not accessible aur top two states mein rightmost column hain terminal, that is, they end episode.
- **Actions**: Movements par each step, that is, up, down, left, and right. The environment hai randomized so that actions can have unintended outcomes. For each action, there hai ek 80% probability to move to expected state, and 10% probability to move mein ek adjacent direction (Udaharan ke liye, right ya left instead ka up ya up ya down instead ka right).
- **Rewards**: As depicted mein right-hand side panel, each state results in -.02, except ke liye the +1/-1 rewards mein terminal states:

<img src="../assets/mdp.png" width="500">

The right panel ka preceding GridWorld diagram shows optimal value estimate that's produced by Value Iteration aur corresponding greedy policy. The negative rewards, combined ke saath uncertainty mein environment, produce ek optimal policy that involves moving away se negative terminal state.

The results hain sensitive to both rewards aur discount factor. The cost ka negative state affects policy mein surrounding fields, and you should modify example mein corresponding notebook to identify threshold levels that alter optimal action selection.

## Imports & Settings

In [1]:
%matplotlib inline

from time import process_time
import numpy as np
import pandas as pd
from mdptoolbox import mdp
from itertools import product

## Set up Gridworld

### States, Actions aur Rewards

We will begin by defining environment parameters:

In [2]:
grid_size = (3, 4)
blocked_cell = (1, 1)
baseline_reward = -0.02
absorbing_cells = {(0, 3): 1, (1, 3): -1}

In [3]:
actions = ['L', 'U', 'R', 'D']
num_actions = len(actions)
probs = [.1, .8, .1, 0]

We will frequently need to convert ke beech one-dimensional aur two-dimensional representations, so we will define two helper functions ke liye this purpose; states hain one-dimensional aur cells hain corresponding two-dimensional coordinates:

In [4]:
to_1d = lambda x: np.ravel_multi_index(x, grid_size)
to_2d = lambda x: np.unravel_index(x, grid_size)

Furthermore, we will precompute some data points to make code more concise:

In [5]:
num_states = np.product(grid_size)
cells = list(np.ndindex(grid_size))
states = list(range(len(cells)))

In [6]:
cell_state = dict(zip(cells, states))
state_cell= dict(zip(states, cells))

In [7]:
absorbing_states = {to_1d(s):r for s, r in absorbing_cells.items()}
blocked_state = to_1d(blocked_cell)

We store rewards ke liye each state:

In [8]:
state_rewards = np.full(num_states, baseline_reward)
state_rewards[blocked_state] = 0
for state, reward in absorbing_states.items():
    state_rewards[state] = reward

In [9]:
action_outcomes = {}
for i, action in enumerate(actions):
    probs_ = dict(zip([actions[j % 4] for j in range(i, num_actions + i)], probs))
    action_outcomes[actions[(i + 1) % 4]] = probs_

To account ke liye probabilistic environment, we also need to compute probability distribution over actual move ke liye ek given action:

In [10]:
action_outcomes

{'U': {'L': 0.1, 'U': 0.8, 'R': 0.1, 'D': 0},
 'R': {'U': 0.1, 'R': 0.8, 'D': 0.1, 'L': 0},
 'D': {'R': 0.1, 'D': 0.8, 'L': 0.1, 'U': 0},
 'L': {'D': 0.1, 'L': 0.8, 'U': 0.1, 'R': 0}}

Now, we hain ready to compute transition matrix, which hai key input to MDP.

### Transition Matrix

The transition matrix defines probability to end up mein ek certain state, S, for each previous state aur action, A, $P(s^\prime \mid s, a)$. We will demonstrate `pymdptoolbox`, and use one ka formats that's available to us to specify transitions aur rewards. For both transition probabilities, we will create karein a `NumPy` array ke saath dimensions of $A \times S \times S$.

First, we compute target cell ke liye each starting cell aur move:

In [11]:
def get_new_cell(state, move):
    cell = to_2d(state)
    if actions[move] == 'U':
        return cell[0] - 1, cell[1]
    elif actions[move] == 'D':
        return cell[0] + 1, cell[1]
    elif actions[move] == 'R':
        return cell[0], cell[1] + 1
    elif actions[move] == 'L':
        return cell[0], cell[1] - 1

In [12]:
state_rewards

array([-0.02, -0.02, -0.02,  1.  , -0.02,  0.  , -0.02, -1.  , -0.02,
       -0.02, -0.02, -0.02])

The following function uses argument's starting `state`, `action`, and `outcome` to fill mein transition probabilities aur rewards:

In [13]:
def update_transitions_and_rewards(state, action, outcome):
    if state in absorbing_states.keys() or state == blocked_state:
        transitions[action, state, state] = 1
    else:
        new_cell = get_new_cell(state, outcome)
        p = action_outcomes[actions[action]][actions[outcome]]
        if new_cell not in cells or new_cell == blocked_cell:
            transitions[action, state, state] += p
            rewards[action, state, state] = baseline_reward
        else:
            new_state= to_1d(new_cell)
            transitions[action, state, new_state] = p
            rewards[action, state, new_state] = state_rewards[new_state]

We generate transition aur reward values by creating placeholder data structures aur iterating over Cartesian product of $A \times S \times S$, as follows:

In [14]:
rewards = np.zeros(shape=(num_actions, num_states, num_states))
transitions = np.zeros((num_actions, num_states, num_states))
actions_ = list(range(num_actions))
for action, outcome, state in product(actions_, actions_, states):
    update_transitions_and_rewards(state, action, outcome)

In [15]:
rewards.shape, transitions.shape

((4, 12, 12), (4, 12, 12))

## PyMDPToolbox

We can also solve MDPs using the [pymdptoolbox](https://pymdptoolbox.readthedocs.io/en/latest/api/mdptoolbox.html) Python library, which includes ek few more algorithms, including Q-learning.

### Value Iteration

In [16]:
gamma = .99
epsilon = 1e-5

To run `ValueIteration`, just instantiate corresponding object ke saath desired configuration options aur rewards aur transition matrices before calling the `.run()` method:

In [17]:
vi = mdp.ValueIteration(transitions=transitions,
                        reward=rewards,
                        discount=gamma,
                        epsilon=epsilon)

vi.run()
f'# Iterations: {vi.iter:,d} | Time: {vi.time:.4f}'

'# Iterations: 31 | Time: 0.0006'

In [18]:
policy = np.asarray([actions[i] for i in vi.policy])
pd.DataFrame(policy.reshape(grid_size))

,0,1,2,3
0,R,R,R,L
1,U,L,U,L
2,U,L,L,L


In [19]:
value = np.asarray(vi.V).reshape(grid_size)
pd.DataFrame(value)

,0,1,2,3
0,0.884143,0.925054,0.961986,0.000000
1,0.848181,0.000000,0.714643,0.000000
2,0.808345,0.773328,0.736099,0.516083


### Policy Iteration

The `PolicyIteration` function works similarly:

In [20]:
pi = mdp.PolicyIteration(transitions=transitions,
                        reward=rewards,
                        discount=gamma,
                        max_iter=1000)

pi.run()
f'# Iterations: {pi.iter:,d} | Time: {pi.time:.4f}'

'# Iterations: 7 | Time: 0.0560'

It also yields same policy, but value function varies by run aur does not need to achieve optimal value before policy converges.

In [21]:
policy = np.asarray([actions[i] for i in pi.policy])
pd.DataFrame(policy.reshape(grid_size))

,0,1,2,3
0,R,R,R,L
1,U,L,U,L
2,U,L,L,L


In [22]:
value = np.asarray(pi.V).reshape(grid_size)
pd.DataFrame(value)

,0,1,2,3
0,0.884143,0.925054,0.961986,-1.389785e-16
1,0.848181,0.000000,0.714643,5.749281e-16
2,0.808345,0.773328,0.736099,5.160828e-01


## Value Iteration

In [23]:
skip_states = list(absorbing_states.keys())+[blocked_state]
states_to_update = [s for s in states if s not in skip_states]

Then, we initialize value function aur set discount factor gamma aur convergence threshold epsilon:

In [24]:
V = np.random.rand(num_states)
V[skip_states] = 0

In [25]:
gamma = .99
epsilon = 1e-5

The algorithm updates value function using Bellman optimality equation, and terminates when L1 norm ka V changes less than epsilon mein absolute terms:

In [26]:
iterations = 0
start = process_time()
converged = False
while not converged:
    V_ = np.copy(V)
    for state in states_to_update:
        q_sa = np.sum(transitions[:, state] * (rewards[:, state] + gamma* V), axis=1)
        V[state] = np.max(q_sa)
    if np.sum(np.fabs(V - V_)) < epsilon:
        converged = True

    iterations += 1
    if iterations % 1000 == 0:
        print(np.sum(np.fabs(V - V_)))

f'# Iterations {iterations} | Time {process_time() - start:.4f}'

'# Iterations 24 | Time 0.0023'

### Value Function

In [27]:
print(pd.DataFrame(V.reshape(grid_size)))

          0         1         2         3
0  0.884143  0.925054  0.961986  0.000000
1  0.848181  0.000000  0.714643  0.000000
2  0.808345  0.773328  0.736099  0.516083


In [28]:
np.allclose(V.reshape(grid_size), np.asarray(vi.V).reshape(grid_size))

True

### Optimal Policy

In [29]:
for state, reward in absorbing_states.items():
    V[state] = reward

policy = np.argmax(np.sum(transitions * V, 2),0)
policy

array([2, 2, 2, 0, 1, 0, 0, 0, 1, 0, 0, 0])

In [30]:
pd.DataFrame(policy.reshape(grid_size)).replace(dict(enumerate(actions)))

,0,1,2,3
0,R,R,R,L
1,U,L,L,L
2,U,L,L,L


## Policy Iteration

Policy iterations involves separate evaluation aur improvement steps. Hum define karte hain improvement part by selecting action that maximizes sum ka expected reward aur next-state value. Note that we temporarily fill mein rewards ke liye terminal states to avoid ignoring actions that would lead us there:

In [31]:
def policy_improvement(value, transitions):
    for state, reward in absorbing_states.items():
        value[state] = reward
    return np.argmax(np.sum(transitions * value, 2),0)

In [32]:
V = np.random.rand(num_states)
V[skip_states] = 0
pi = np.random.choice(list(range(num_actions)), size=num_states)

The algorithm alternates ke beech policy evaluation ke liye ek greedily selected action aur policy improvement until policy stabilizes:

In [33]:
iterations = 0
start = process_time()
converged = False
while not converged:
    pi_ = np.copy(pi)
    for state in states_to_update:
        action = policy[state]
        V[state] = np.dot(transitions[action, state], (rewards[action, state] + gamma* V))
        pi = policy_improvement(V.copy(), transitions)
    if np.array_equal(pi_, pi):
        converged = True
    iterations += 1

f'# Iterations {iterations} | Time {process_time() - start:.4f}'

'# Iterations 3 | Time 0.0009'

Policy iteration converges after only three iterations. The policy stabilizes before algorithm finds optimal value function, and optimal policy differs slightly, most notably by suggesting up instead ka safer left ke liye field next to negative terminal state. This can be avoided by tightening convergence criteria, Udaharan ke liye, by requiring ek stable policy ka several rounds ya adding ek threshold ke liye value function.

In [34]:
pd.DataFrame(pi.reshape(grid_size)).replace(dict(enumerate(actions)))

,0,1,2,3
0,R,R,R,L
1,U,L,U,L
2,U,L,L,L


In [35]:
pd.DataFrame(V.reshape(grid_size))

,0,1,2,3
0,0.756333,0.882232,0.933790,0.000000
1,0.683594,0.000000,0.480169,0.000000
2,0.612364,0.552599,0.506767,0.307299
